# Lab Day 1 — Lê Nguyễn Trâm Anh — 2A202602760
Notebook chạy theo stage trên Kaggle GPU. Mỗi run được lưu ngay; chạy lại cell sẽ resume nếu fingerprint cấu hình khớp. **Không xem eval trước Stage 5.**

In [1]:
# Stage 0 — Bootstrap Kaggle/local
from pathlib import Path
import gzip
import json
import os
import shutil
import subprocess
import sys
import zipfile

IS_KAGGLE = Path("/kaggle/input").exists()
MSSV = "2A202602760"

if IS_KAGGLE:
    input_root = Path("/kaggle/input")

    # Kaggle Dataset có thể chứa CSV thường hoặc CSV nén.
    candidates = (
        list(input_root.glob("**/data/covtype.csv.gz"))
        or list(input_root.glob("**/data/covtype.csv"))
    )

    # Trường hợp Dataset chứa ZIP chưa được giải nén.
    if not candidates:
        bundles = list(input_root.glob(f"**/kaggle_bundle_{MSSV}.zip"))
        if not bundles:
            raise FileNotFoundError(
                "Không tìm thấy data/covtype.csv(.gz) hoặc "
                f"kaggle_bundle_{MSSV}.zip trong /kaggle/input"
            )

        unpacked = Path("/kaggle/working/_input_bundle")
        unpacked.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(bundles[0]) as zf:
            zf.extractall(unpacked)

        candidates = (
            list(unpacked.glob("**/data/covtype.csv.gz"))
            or list(unpacked.glob("**/data/covtype.csv"))
        )
        if not candidates:
            raise FileNotFoundError(
                "Đã giải nén bundle nhưng không thấy data/covtype.csv(.gz)"
            )

    source_root = candidates[0].parents[1]
    REPO_ROOT = Path("/kaggle/working/neuralnetwork_lab")

    if not REPO_ROOT.exists():
        shutil.copytree(
            source_root,
            REPO_ROOT,
            ignore=shutil.ignore_patterns(
                ".git", "__pycache__", ".ipynb_checkpoints"
            ),
        )
else:
    here = Path.cwd().resolve()
    REPO_ROOT = next(
        (
            p for p in (here, *here.parents)
            if (p / "data/covtype.csv.gz").exists()
            or (p / "data/covtype.csv").exists()
        ),
        None,
    )
    if REPO_ROOT is None:
        raise FileNotFoundError("Không tìm thấy repo chứa data/covtype.csv(.gz)")

# Script split_data.py dùng đường dẫn covtype.csv.gz.
# Chỉ tạo bản nén trong thư mục làm việc, không sửa Kaggle Input.
csv_path = REPO_ROOT / "data/covtype.csv"
gz_path = REPO_ROOT / "data/covtype.csv.gz"

if csv_path.exists() and not gz_path.exists():
    print("Đang tạo data/covtype.csv.gz từ data/covtype.csv ...")
    with csv_path.open("rb") as src, gzip.open(gz_path, "wb") as dst:
        shutil.copyfileobj(src, dst)

SUB = REPO_ROOT / f"submission_{MSSV}"
RESULTS = SUB / "results"
FIGURES = SUB / "figures"
CHECKPOINTS = REPO_ROOT / "runtime_checkpoints"

for path in (SUB, RESULTS, FIGURES, CHECKPOINTS, SUB / "code"):
    path.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT / "code"))
os.chdir(REPO_ROOT)

# Khôi phục kết quả nếu đã attach archive của phiên Kaggle trước.
if IS_KAGGLE and not any(RESULTS.glob("*.json")):
    archives = list(Path("/kaggle/input").glob("**/artifacts_stage*.zip"))

    if archives:
        def archive_score(path):
            with zipfile.ZipFile(path) as zf:
                return sum(
                    name.startswith("results/") and name.endswith(".json")
                    for name in zf.namelist()
                )

        newest = max(archives, key=archive_score)
        restore = REPO_ROOT / "_restore"
        restore.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(newest) as zf:
            zf.extractall(restore)

        for name, target in (
            ("results", RESULTS),
            ("figures", FIGURES),
            ("runtime_checkpoints", CHECKPOINTS),
        ):
            src = restore / name
            if src.exists():
                shutil.copytree(src, target, dirs_exist_ok=True)

        for name in ("health_checks.json", "selection_state.json"):
            src = restore / name
            if src.exists():
                shutil.copy2(src, SUB / name)

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if IS_KAGGLE and DEVICE != "cuda":
    raise RuntimeError("Chưa bật GPU: Kaggle → Settings → Accelerator → GPU")

print("Nguồn dữ liệu:", source_root if IS_KAGGLE else REPO_ROOT)
print("Repo làm việc:", REPO_ROOT)
print("CSV nén sẵn sàng:", gz_path.exists())
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Đang tạo data/covtype.csv.gz từ data/covtype.csv ...
Nguồn dữ liệu: /kaggle/input/datasets/nguyentramanhle/lenguyentramanh
Repo làm việc: /kaggle/working/neuralnetwork_lab
CSV nén sẵn sàng: True
Device: cuda
PyTorch: 2.11.0+cu128
GPU: Tesla T4


In [2]:
from pathlib import Path

files = list(Path("/kaggle/input").glob("**/*"))
print("\n".join(map(str, files[:100])))

/kaggle/input/datasets
/kaggle/input/datasets/nguyentramanhle
/kaggle/input/datasets/nguyentramanhle/lenguyentramanh
/kaggle/input/datasets/nguyentramanhle/day01-track4
/kaggle/input/datasets/nguyentramanhle/lenguyentramanh/code
/kaggle/input/datasets/nguyentramanhle/lenguyentramanh/GUIDE.md
/kaggle/input/datasets/nguyentramanhle/lenguyentramanh/scripts
/kaggle/input/datasets/nguyentramanhle/lenguyentramanh/README.md
/kaggle/input/datasets/nguyentramanhle/lenguyentramanh/KAGGLE_RUN.md
/kaggle/input/datasets/nguyentramanhle/lenguyentramanh/templates
/kaggle/input/datasets/nguyentramanhle/lenguyentramanh/RUBRIC.md
/kaggle/input/datasets/nguyentramanhle/lenguyentramanh/data
/kaggle/input/datasets/nguyentramanhle/day01-track4/code
/kaggle/input/datasets/nguyentramanhle/day01-track4/GUIDE.md
/kaggle/input/datasets/nguyentramanhle/day01-track4/scripts
/kaggle/input/datasets/nguyentramanhle/day01-track4/README.md
/kaggle/input/datasets/nguyentramanhle/day01-track4/KAGGLE_RUN.md
/kaggle/input/

## Stage 1 — Dữ liệu và health checks
Kỳ vọng: loss bước 0 gần `ln(7)`, logits `(B,7)`, đúng 47.879 tham số, mọi gradient khác 0 và overfit được 20 mẫu.

In [3]:
from data import prepare_data
from workflow import run_health_checks,save_stage_archive
processed=REPO_ROOT/'data/processed'
if not (processed/'train.npz').exists(): subprocess.run([sys.executable,'scripts/split_data.py'],cwd=REPO_ROOT,check=True)
data=prepare_data(DEVICE,0.2,42,str(processed))
assert torch.allclose(data['X_tr'][:,:10].mean(0),torch.zeros(10,device=DEVICE),atol=2e-4)
assert torch.allclose(data['X_tr'][:,:10].std(0,unbiased=False),torch.ones(10,device=DEVICE),atol=2e-4)
health=run_health_checks(data,FIGURES/'health_overfit20.png')
(SUB/'health_checks.json').write_text(json.dumps(health,indent=2),encoding='utf-8')
display(health); print(save_stage_archive('stage1',SUB,CHECKPOINTS,REPO_ROOT))

train: X (464809, 54) float32, y (464809,) int64, nhãn 0..6 -> data/processed/train.npz
eval : X (116203, 54) float32, y (116203,) int64, nhãn 0..6 -> data/processed/eval.npz

lớp   train(%)   eval(%)
  0    36.461    36.460
  1    48.760    48.760
  2     6.154     6.154
  3     0.473     0.472
  4     1.634     1.634
  5     2.989     2.989
  6     3.530     3.530

đoán luôn lớp đa số (lớp 1) cho accuracy trên eval = 0.4876
train=371,847, val=92,962, eval=116,203; majority_class=1, val_majority_acc=0.4876


{'parameter_count': 47879,
 'logits_shape': [8, 7],
 'step0_loss': 2.3775719671999713,
 'ln7': 1.9459101490553132,
 'gradient_norms': {'net.0.weight': 0.9154943227767944,
  'net.0.bias': 0.41088810563087463,
  'net.2.weight': 3.0234246253967285,
  'net.2.bias': 0.5389723181724548,
  'net.4.weight': 2.491093397140503,
  'net.4.bias': 0.6510047912597656},
 'overfit_steps': 20,
 'overfit_final_loss': 0.0008133966475725174,
 'overfit_final_acc': 1.0}

/kaggle/working/neuralnetwork_lab/artifacts_stage1.zip


## Stage 2 — Baseline, learning-rate sweep và nhiễu seed
Dự đoán: lr quá nhỏ hội tụ chậm, lr quá lớn dao động; chỉ chọn lr bằng validation. Ba seed dùng để tính `2σ`.

In [4]:
from workflow import baseline_sweep_configs,official_baseline_configs,run_many,select_best,seed_statistics
sweep=run_many(baseline_sweep_configs(),data,RESULTS,FIGURES,CHECKPOINTS)
selected_lr=float(select_best(sweep)['cfg']['lr'])
baseline_runs=run_many(official_baseline_configs(selected_lr),data,RESULTS,FIGURES,CHECKPOINTS)
baseline_stats=seed_statistics(baseline_runs)
state={'selected_lr':selected_lr,'baseline_ids':[r['cfg']['exp_id'] for r in baseline_runs],'baseline_stats':baseline_stats}
(SUB/'selection_state.json').write_text(json.dumps(state,indent=2),encoding='utf-8')
display(state); print(save_stage_archive('stage2',SUB,CHECKPOINTS,REPO_ROOT))

[run] base-lr0p01-s1 | Dò learning rate baseline: 0.01


/kaggle/working/neuralnetwork_lab/code/train.py:247: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(cfg["precision"] == "fp16"))


[run] base-lr0p03-s1 | Dò learning rate baseline: 0.03
[run] base-lr0p1-s1 | Dò learning rate baseline: 0.1
[run] base-s1 | Baseline chính thức, seed 1
[run] base-s2 | Baseline chính thức, seed 2
[run] base-s3 | Baseline chính thức, seed 3


{'selected_lr': 0.1,
 'baseline_ids': ['base-s1', 'base-s2', 'base-s3'],
 'baseline_stats': {'n': 3,
  'f1_mean': 0.856226115685378,
  'f1_std': 0.0018587755051709784,
  'acc_mean': 0.9090847156185683,
  'acc_std': 0.0006520543818068699,
  'noise_2sigma': 0.003717551010341957}}

/kaggle/working/neuralnetwork_lab/artifacts_stage2.zip


## Stage 3 — Bảy nhóm thí nghiệm
Loss: CE được kỳ vọng ổn định hơn MSE; class weighting có thể giúp lớp hiếm. Optimizer chỉ được so sau khi mỗi họ được dò ba lr. Batch làm đổi số update; dropout chỉ hữu ích khi overfit. Clipping phải được kiểm chứng bằng grad norm ở lr cao. FP16 có thể giảm memory nhưng chưa chắc nhanh. Zeros dự kiến hỏng do đối xứng.

In [5]:
from workflow import optimizer_configs,topic_configs,make_comparison_figures
state=json.loads((SUB/'selection_state.json').read_text(encoding='utf-8')); base_cfg=official_baseline_configs(state['selected_lr'])[0]
optimizer_runs=run_many(optimizer_configs(base_cfg),data,RESULTS,FIGURES,CHECKPOINTS)
base_s1=baseline_runs[0] if 'baseline_runs' in globals() else run_many([base_cfg],data,RESULTS,FIGURES,CHECKPOINTS)[0]
clip_c=float(base_s1['summary']['grad_norm_p75']); bf16_ok=bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())
groups=topic_configs(base_cfg,clip_c,bf16_ok)
for group_name,configs in groups.items():
    print('===',group_name,'==='); run_many(configs,data,RESULTS,FIGURES,CHECKPOINTS)
    print(save_stage_archive(f'stage3_{group_name}',SUB,CHECKPOINTS,REPO_ROOT))
make_comparison_figures(RESULTS,FIGURES)

[run] opt-sgd-lr0p01 | sgd với lr=0.01
[run] opt-sgd-lr0p03 | sgd với lr=0.03
[run] opt-sgd-lr0p1 | sgd với lr=0.1
[run] opt-adam-lr0p0003 | adam với lr=0.0003
[run] opt-adam-lr0p001 | adam với lr=0.001
[run] opt-adam-lr0p003 | adam với lr=0.003
[run] opt-adamw-lr0p0003 | adamw với lr=0.0003
[run] opt-adamw-lr0p001 | adamw với lr=0.001
[run] opt-adamw-lr0p003 | adamw với lr=0.003
=== loss ===
[run] loss-mse | MSE với nhãn one-hot
[run] loss-ce-balanced | CE trọng số nghịch đảo tần suất
[run] loss-ce-sqrt | CE trọng số căn nghịch đảo tần suất
/kaggle/working/neuralnetwork_lab/artifacts_stage3_loss.zip
=== hparam ===
[run] batch-128 | Batch size 128
[run] batch-2048 | Batch size 2048
[run] arch-wide | M-wide 54-512-256-7
[run] arch-deep | M-deep 54-256-128-64-7
[run] epochs-40 | Baseline 40 epoch
/kaggle/working/neuralnetwork_lab/artifacts_stage3_hparam.zip
=== dropout ===
[run] drop-0p1 | Dropout q=0.1
[run] drop-0p3 | Dropout q=0.3
[run] drop-0p5 | Dropout q=0.5
/kaggle/working/neuraln

## Stage 4 — Hai candidate, mỗi candidate ba seed
Candidate chỉ dùng validation. Nếu chênh lệch mean không vượt `max(2σ,0.002)`, ưu tiên ổn định, đơn giản và thời gian.

In [6]:
from results_table import load_results
from workflow import build_candidate_configs,expand_candidate_seeds,choose_final_candidate
state=json.loads((SUB/'selection_state.json').read_text(encoding='utf-8')); all_results=load_results(RESULTS); base_cfg=official_baseline_configs(state['selected_lr'])[0]
candidate_a,candidate_b=build_candidate_configs(all_results,base_cfg,state['baseline_stats']['noise_2sigma'])
a_runs=run_many(expand_candidate_seeds(candidate_a,'a'),data,RESULTS,FIGURES,CHECKPOINTS)
b_runs=run_many(expand_candidate_seeds(candidate_b,'b'),data,RESULTS,FIGURES,CHECKPOINTS)
winner,chosen_runs,candidate_stats=choose_final_candidate(a_runs,b_runs,state['baseline_stats']['noise_2sigma'])
state.update({'winner':winner,'final_ids':[r['cfg']['exp_id'] for r in chosen_runs],'candidate_stats':candidate_stats})
(SUB/'selection_state.json').write_text(json.dumps(state,indent=2),encoding='utf-8'); make_comparison_figures(RESULTS,FIGURES)
display(state); print(save_stage_archive('stage4',SUB,CHECKPOINTS,REPO_ROOT))

[run] candidate-a-s1 | Candidate A, seed 1
[run] candidate-a-s2 | Candidate A, seed 2
[run] candidate-a-s3 | Candidate A, seed 3
[run] candidate-b-s1 | Candidate B, seed 1
[run] candidate-b-s2 | Candidate B, seed 2
[run] candidate-b-s3 | Candidate B, seed 3


{'selected_lr': 0.1,
 'baseline_ids': ['base-s1', 'base-s2', 'base-s3'],
 'baseline_stats': {'n': 3,
  'f1_mean': 0.856226115685378,
  'f1_std': 0.0018587755051709784,
  'acc_mean': 0.9090847156185683,
  'acc_std': 0.0006520543818068699,
  'noise_2sigma': 0.003717551010341957},
 'winner': 'b',
 'final_ids': ['candidate-b-s1', 'candidate-b-s2', 'candidate-b-s3'],
 'candidate_stats': {'candidate_a': {'n': 3,
   'f1_mean': 0.8684230889968735,
   'f1_std': 0.006899083757731137,
   'acc_mean': 0.9122723980407765,
   'acc_std': 0.001687194058194079,
   'noise_2sigma': 0.013798167515462274},
  'candidate_b': {'n': 3,
   'f1_mean': 0.8846373400440241,
   'f1_std': 0.002368476228899568,
   'acc_mean': 0.9253924542644665,
   'acc_std': 0.0008076652524859839,
   'noise_2sigma': 0.004736952457799136},
  'threshold': 0.003717551010341957}}

/kaggle/working/neuralnetwork_lab/artifacts_stage4.zip


## Stage 5 — Khóa lựa chọn rồi mới đánh giá eval
Cell tạo prediction baseline/final seed 1, gọi đúng `scripts/evaluate.py`, rồi sinh Excel và báo cáo. Không thay đổi cấu hình sau khi bắt đầu.

In [7]:
from train import final_eval as write_eval_predictions
from reporting import build_xlsx,generate_report,read_json
state=json.loads((SUB/'selection_state.json').read_text(encoding='utf-8')); base_cfg=official_baseline_configs(state['selected_lr'])[0]
base_result=run_many([base_cfg],data,RESULTS,FIGURES,CHECKPOINTS)[0]
all_results=load_results(RESULTS); candidate_a,candidate_b=build_candidate_configs(all_results,base_cfg,state['baseline_stats']['noise_2sigma'])
winner_cfg=candidate_a if state['winner']=='a' else candidate_b; winner_cfg=expand_candidate_seeds(winner_cfg,state['winner'])[0]
final_result=run_many([winner_cfg],data,RESULTS,FIGURES,CHECKPOINTS)[0]
baseline_pred=REPO_ROOT/'baseline_predictions_eval.csv'; baseline_json=REPO_ROOT/'baseline_eval_result.json'
write_eval_predictions(base_result['cfg'],base_result,data,str(baseline_pred))
subprocess.run([sys.executable,'scripts/evaluate.py','--pred',str(baseline_pred),'--out',str(baseline_json)],cwd=REPO_ROOT,check=True)
write_eval_predictions(final_result['cfg'],final_result,data,str(SUB/'predictions_eval.csv'))
subprocess.run([sys.executable,'scripts/evaluate.py','--pred',str(SUB/'predictions_eval.csv'),'--out',str(SUB/'eval_result.json')],cwd=REPO_ROOT,check=True)
baseline_eval,final_scores=read_json(baseline_json),read_json(SUB/'eval_result.json'); all_results=load_results(RESULTS)
build_xlsx(all_results,REPO_ROOT/'templates/experiment_table_template.xlsx',SUB/'experiments.xlsx',state['baseline_ids'],state['final_ids'],baseline_eval,final_scores)
generate_report(all_results,read_json(SUB/'health_checks.json'),state['baseline_ids'],state['final_ids'],baseline_eval,final_scores,state['winner'],SUB/'REPORT.md')
print('FINAL EVAL:',final_scores['accuracy'],final_scores['macro_f1'])

[resume] base-s1
[resume] candidate-b-s1
n_eval = 116203
accuracy = 0.9064
macro_f1 = 0.8618   <- chỉ số chính

lớp  support  precision  recall    f1
  0    42368     0.9115  0.8912  0.9012
  1    56661     0.9092  0.9332  0.9211
  2     7151     0.9008  0.8990  0.8999
  3      549     0.8381  0.8015  0.8194
  4     1899     0.8357  0.7125  0.7692
  5     3473     0.8275  0.7777  0.8018
  6     4102     0.9230  0.9171  0.9200

ma trận nhầm lẫn (hàng = thật, cột = dự đoán):
       0      1     2    3     4     5     6
0  37760   4264     5    0    49     9   281
1   3338  52876   118    0   200    96    33
2      0    237  6429   65    12   408     0
3      0      2    65  440     0    42     0
4     28    483    27    0  1353     8     0
5      2    253   493   20     4  2701     0
6    300     39     0    0     1     0  3762

đã ghi /kaggle/working/neuralnetwork_lab/baseline_eval_result.json
n_eval = 116203
accuracy = 0.9227
macro_f1 = 0.8860   <- chỉ số chính

lớp  support  precision

## Stage 6 — Đồng bộ code, validate và đóng ZIP
ZIP chỉ được tạo khi hợp lệ; checkpoint và dữ liệu không được đưa vào bài nộp.

In [8]:
for source in (REPO_ROOT/'code').iterdir():
    if source.is_file() and source.suffix in {'.py','.ipynb','.txt'}: shutil.copy2(source,SUB/'code'/source.name)
from validate_submission import validate_submission,create_zip
errors=validate_submission(SUB,REPO_ROOT/'data/processed/eval.npz')
if errors: raise AssertionError('SUBMISSION CHƯA HỢP LỆ:\n- '+'\n- '.join(errors))
zip_path=create_zip(SUB,REPO_ROOT/'submission_2A202602760.zip')
print('SUBMISSION HỢP LỆ:',zip_path); print('Tải ZIP từ Kaggle Output; không chỉnh cấu hình sau khi xem eval.')

SUBMISSION HỢP LỆ: /kaggle/working/neuralnetwork_lab/submission_2A202602760.zip
Tải ZIP từ Kaggle Output; không chỉnh cấu hình sau khi xem eval.
